# Trayectoria y agrupamiento de circuitos con ventana deslizante

Cuaderno hermano de `01.2_uiti_vano_kmeans.ipynb`, con el mismo esquema: todo se precomputa en
Python y un panel HTML+JS maneja la figura sin recalcular nada.

Aca la unidad no es el circuito sino el par **circuito x ventana**. Cada mes aporta dos ventanas
-- el **mes calendario completo** y la **cruzada**, del dia 15 de ese mes al 15 del siguiente --
y al ordenarlas por fecha de inicio quedan alternadas, con paso efectivo de medio mes:

| ventana | periodo | que cubre |
|---|---|---|
| V1 | 2025-11-01 a 2025-11-30 | mes 1 completo |
| V2 | 2025-11-15 a 2025-12-14 | cruzada: mes 1 del 15 a fin + mes 2 del 1 al 14 |
| V3 | 2025-12-01 a 2025-12-31 | mes 2 completo |
| V4 | 2025-12-15 a 2026-01-14 | cruzada |
| V5 | 2026-01-01 a 2026-01-31 | mes 3 completo |
| ... | ... | ... |

Con seis meses de base salen **11 ventanas**. Para cada par circuito x ventana se acumula el UITI
(`UITI_VANO`) y se cuenta el numero de eventos dentro de esa ventana.

Sobre ese plano corre un **K-Means a 4 grupos**, igual que en `01.2`: la unidad que se agrupa es
el par circuito x ventana, asi que un mismo circuito puede caer en grupos distintos segun la
ventana. Los grupos se nombran por el **ranking de la mediana del UITI acumulado**
(`Bajo`, `Medio`, `Medio-Alto`, `Alto`).

El panel trae:

- **Circuito**: al elegir uno, sus ventanas se resaltan y se **conectan con flechas** en orden
  cronologico, dibujando su trayectoria. El tooltip de cada punto indica el rango de fechas de
  su ventana y el grupo en que cayo.
- **Log eje X**, **Log eje Y** y **Preproceso**: igual que en `01.2`, se aplican *antes* de correr
  K-Means, asi que cambian la particion y no solo el dibujo.
- **Descargar tabla (CSV)**: baja la tabla completa. La ultima celda hace lo mismo desde Python.

Ademas del mapa, el tablero muestra por grupo el **numero de vanos unicos** que lo componen y los
**violines** del UITI acumulado y del numero de eventos.

> **Todas las ventanas consecutivas se solapan**, entre 14 y 17 dias segun el largo del mes. Ese
> solape es lo que hace que la trayectoria se mueva suave: dos puntos vecinos comparten cerca de la
> mitad de sus datos, asi que un salto grande entre ellos indica un evento fuerte concentrado en
> los dias que **no** comparten. Las 11 ventanas cubren el 100% de los eventos de la base.

> **Huecos.** Alrededor del 76% de los pares circuito x ventana registra algun evento. La **tabla**
> lleva la grilla completa, con ceros donde no hubo eventos, para que sea regular aguas abajo. El
> **mapa y el agrupamiento** usan solo las celdas con al menos un evento, porque un cero no tiene
> lugar en un eje logaritmico; cuando un circuito se saltea una ventana, la flecha une las dos
> presentes mas cercanas.

In [1]:
# Descomentar solo si el entorno no tiene instaladas estas dependencias.
# %pip install pandas numpy plotly

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler

NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
# Paleta Reds, de claro a oscuro, para que el color ya ordene los grupos por criticidad.
COLORES_GRUPOS = ['rgb(252,187,161)', 'rgb(251,106,74)', 'rgb(203,24,29)', 'rgb(103,0,13)']
PREPROCESOS = {'minmax': MinMaxScaler, 'zscore': StandardScaler}
SIN_SELECCION = '(ninguno)'
SEMILLA = 42
# Opacidad de la nube de fondo: la de siempre y la atenuada, que entra cuando hay un
# circuito elegido para que su trayectoria no compita con los otros 207.
OPACIDAD_NUBE = 0.55
OPACIDAD_NUBE_ATENUADA = 0.10


# Sube desde el cwd hasta encontrar data/Indicadores_vano_v3.csv, para que el cuaderno
# funcione sin importar desde que directorio se ejecute (Jupyter local o Colab/Kaggle).
def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data/Indicadores_vano_v3.csv').exists():
            return candidate
    raise FileNotFoundError('No se encontro data/Indicadores_vano_v3.csv subiendo desde el cwd')


REPO_ROOT = find_repo_root()

# FID_VANO hace falta para contar vanos unicos por grupo; el CSV completo trae ~270 columnas.
df = pd.read_csv(REPO_ROOT / 'data' / 'Indicadores_vano_v3.csv',
                 usecols=['CIRCUITO', 'FID_VANO', 'UITI_VANO', 'FECHA'])
df['FECHA'] = pd.to_datetime(df['FECHA'], errors='coerce')
df['UITI_VANO'] = pd.to_numeric(df['UITI_VANO'], errors='coerce').fillna(0.0)
# FID_VANO llega numerico con sufijo '.0' inconsistente entre filas; se normaliza como string
# igual que `chec_local_interpreter.plotting._norm_map_id`, para no duplicar vanos por formato.
df['FID_VANO'] = df['FID_VANO'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)

CIRCUITOS = sorted(df['CIRCUITO'].unique())
ESPACIOS = [(lx, ly, prep)
            for lx in (False, True) for ly in (False, True)
            for prep in ('minmax', 'zscore')]
IDX_ESPACIO_DEFECTO = ESPACIOS.index((False, False, 'minmax'))

# Cada mes aporta DOS ventanas: el mes calendario completo y la cruzada, que va del dia 15
# de ese mes al 15 del siguiente. Ordenadas por fecha de inicio quedan alternadas
# (mes completo, cruzada, mes completo, ...) y el paso efectivo es de medio mes.
_meses = pd.period_range(df['FECHA'].min(), df['FECHA'].max(), freq='M')
_fin_datos = _meses[-1].to_timestamp(how='end').normalize() + pd.Timedelta(days=1)

# Intervalos semiabiertos [desde, hasta_excl): el dia 15 pertenece a la ventana que arranca
# en el, asi el solape no cuenta dos veces un evento que cae justo en el borde.
_cortes = []
for _k, _m in enumerate(_meses):
    _ini = _m.to_timestamp()
    _fin = _meses[_k + 1].to_timestamp() if _k + 1 < len(_meses) else _fin_datos
    _cortes.append((_ini, _fin))
    _cortes.append((_ini + pd.Timedelta(days=14), _fin + pd.Timedelta(days=14)))
# Una ventana que se pasa del final de la base quedaria corta y dibujaria una caida falsa.
_cortes = sorted(c for c in _cortes if c[1] <= _fin_datos)

VENTANAS = [
    {
        'i': k,
        'desde': desde,
        'hasta_excl': hasta,
        'etiqueta': f'V{k + 1}',
        'periodo': f'{desde.date()} a {(hasta - pd.Timedelta(days=1)).date()}',
    }
    for k, (desde, hasta) in enumerate(_cortes)
]

print(f'{len(df):,} eventos | {len(CIRCUITOS)} circuitos | '
      f'{df["FID_VANO"].nunique():,} vanos | '
      f'{df["FECHA"].min():%Y-%m-%d} a {df["FECHA"].max():%Y-%m-%d}')
for v in VENTANAS:
    _dias = (v['hasta_excl'] - v['desde']).days
    print(f'  {v["etiqueta"]:>3s}  {v["periodo"]}  ({_dias} dias)')

# La ultima media luna de datos no completa una ventana. Se informa en vez de meterla como
# ventana corta: una ventana de la mitad de dias mostraria una caida que no existe.
_cola = VENTANAS[-1]['hasta_excl']
_eventos_cola = int((df['FECHA'] >= _cola).sum())
if _eventos_cola:
    # Un print() vacio en vez de un \n embebido: este texto pasa por un string no-raw del
    # generador y una sola barra se convertiria en salto de linea real, rompiendo la celda.
    print()
    print(f'FUERA DE VENTANA: {_cola.date()} a {df["FECHA"].max():%Y-%m-%d} -- '
          f'{_eventos_cola:,} eventos ({100 * _eventos_cola / len(df):.1f}% de la base) '
          f'no entran en ninguna ventana completa.')

159,470 eventos | 208 circuitos | 27,390 vanos | 2025-11-01 a 2026-04-30
   V1  2025-11-01 a 2025-11-30  (30 dias)
   V2  2025-11-15 a 2025-12-14  (30 dias)
   V3  2025-12-01 a 2025-12-31  (31 dias)
   V4  2025-12-15 a 2026-01-14  (31 dias)
   V5  2026-01-01 a 2026-01-31  (31 dias)
   V6  2026-01-15 a 2026-02-14  (31 dias)
   V7  2026-02-01 a 2026-02-28  (28 dias)
   V8  2026-02-15 a 2026-03-14  (28 dias)
   V9  2026-03-01 a 2026-03-31  (31 dias)
  V10  2026-03-15 a 2026-04-14  (31 dias)
  V11  2026-04-01 a 2026-04-30  (30 dias)


In [3]:
def tabla_ventanas():
    """UITI acumulado y numero de eventos por circuito y ventana, grilla completa.

    Devuelve todas las ventanas para los 208 circuitos, con ceros donde no hubo eventos:
    una grilla regular es mas facil de consumir aguas abajo que una tabla rala.
    """
    piezas = []
    for v in VENTANAS:
        dentro = df[(df['FECHA'] >= v['desde']) & (df['FECHA'] < v['hasta_excl'])]
        agregado = (
            dentro.groupby('CIRCUITO')
            .agg(uiti_acumulado=('UITI_VANO', 'sum'), num_eventos=('UITI_VANO', 'count'))
            .reindex(CIRCUITOS)                       # reindex trae los circuitos ausentes
            .fillna({'uiti_acumulado': 0.0, 'num_eventos': 0})
            .reset_index(names='circuito')
        )
        agregado.insert(1, 'ventana', v['etiqueta'])
        agregado.insert(2, 'desde', str(v['desde'].date()))
        agregado.insert(3, 'hasta', str((v['hasta_excl'] - pd.Timedelta(days=1)).date()))
        piezas.append(agregado)

    tabla = pd.concat(piezas, ignore_index=True)
    tabla['num_eventos'] = tabla['num_eventos'].astype(int)
    tabla['uiti_acumulado'] = tabla['uiti_acumulado'].round(2)
    # Orden circuito -> ventana cronologica: es el orden en que se recorren las flechas.
    tabla['_orden'] = tabla['ventana'].map({v['etiqueta']: v['i'] for v in VENTANAS})
    return (tabla.sort_values(['circuito', '_orden'])
                 .drop(columns='_orden')
                 .reset_index(drop=True))


TABLA = tabla_ventanas()
vacias = int((TABLA['num_eventos'] == 0).sum())
print(f'{len(TABLA)} filas = {len(CIRCUITOS)} circuitos x {len(VENTANAS)} ventanas '
      f'| sin eventos: {vacias} ({100 * vacias / len(TABLA):.1f}%)')
TABLA.head(12)

2288 filas = 208 circuitos x 11 ventanas | sin eventos: 550 (24.0%)


,circuito,ventana,desde,hasta,uiti_acumulado,num_eventos
0,AGU23L12,V1,2025-11-01,2025-11-30,233.75,26
1,AGU23L12,V2,2025-11-15,2025-12-14,233.75,26
2,AGU23L12,V3,2025-12-01,2025-12-31,4092.92,15
3,AGU23L12,V4,2025-12-15,2026-01-14,4092.92,15
4,AGU23L12,V5,2026-01-01,2026-01-31,0.00,0
5,AGU23L12,V6,2026-01-15,2026-02-14,0.00,0
6,AGU23L12,V7,2026-02-01,2026-02-28,0.00,0
7,AGU23L12,V8,2026-02-15,2026-03-14,0.00,0
8,AGU23L12,V9,2026-03-01,2026-03-31,0.00,0
9,AGU23L12,V10,2026-03-15,2026-04-14,416.49,8


In [4]:
# --- celdas del mapa ---------------------------------------------------------------------
POR_CIRCUITO = {
    circuito: {'n': grupo['num_eventos'].tolist(), 'u': grupo['uiti_acumulado'].tolist()}
    for circuito, grupo in TABLA.groupby('circuito', sort=True)
}

# Orden canonico de las celdas con eventos: circuito y despues ventana. El JS reconstruye
# exactamente esta misma secuencia, asi que las etiquetas de grupo se alinean sin enviar
# indices; si los dos recorridos divergieran, los colores del mapa no corresponderian.
CELDAS = [(ci, vi)
          for ci, circuito in enumerate(CIRCUITOS)
          for vi in range(len(VENTANAS))
          if POR_CIRCUITO[circuito]['n'][vi] > 0]
XY = np.array([[POR_CIRCUITO[CIRCUITOS[ci]]['n'][vi],
                POR_CIRCUITO[CIRCUITOS[ci]]['u'][vi]] for ci, vi in CELDAS], dtype=float)

# --- vanos por celda, para contar los unicos de cada grupo -------------------------------
_piezas = []
for v in VENTANAS:
    dentro = df[(df['FECHA'] >= v['desde']) & (df['FECHA'] < v['hasta_excl'])]
    par = dentro[['CIRCUITO', 'FID_VANO']].drop_duplicates()
    par = par.assign(ventana_i=v['i'])
    _piezas.append(par)
CELDA_VANOS = pd.concat(_piezas, ignore_index=True)
_indice_circuito = {c: i for i, c in enumerate(CIRCUITOS)}
CELDA_VANOS['celda'] = (CELDA_VANOS['CIRCUITO'].map(_indice_circuito).astype(int) * len(VENTANAS)
                        + CELDA_VANOS['ventana_i'].astype(int))
_clave_celda = {ci * len(VENTANAS) + vi: k for k, (ci, vi) in enumerate(CELDAS)}


def aplicar_log(X, logs):
    """log10 columna por columna, segun que ejes lo tengan activado."""
    V = np.array(X, dtype=float, copy=True)
    for c, activo in enumerate(logs):
        if activo:
            V[:, c] = np.log10(V[:, c])
    return V


def agrupar_celdas(logs, prep):
    """K-Means a 4 grupos sobre las celdas con eventos, en el espacio ajustado.

    Devuelve las etiquetas y la geometria de la particion (centroides y parametros del
    escalador), que es lo que necesitan los contornos de membresia.
    """
    X = aplicar_log(XY, logs)
    escalador = PREPROCESOS[prep]().fit(X)
    modelo = KMeans(n_clusters=4, random_state=SEMILLA, n_init=10).fit(escalador.transform(X))
    etiquetas = modelo.labels_

    # El id que devuelve K-Means es arbitrario: el nombre del grupo se asigna por el
    # ranking de la MEDIANA del UITI acumulado, de menor a mayor.
    medianas = [np.median(XY[etiquetas == c, 1]) for c in range(4)]
    orden = list(np.argsort(medianas))
    remapeo = {c: i for i, c in enumerate(orden)}

    # Todo escalador se reduce a (v - offset) / scale, asi el JS aplica uno solo.
    if prep == 'minmax':
        offset, scale = escalador.data_min_, escalador.data_range_
    else:
        offset, scale = escalador.mean_, escalador.scale_

    geometria = {
        'logs': [bool(logs[0]), bool(logs[1])],
        'offset': np.round(offset, 6).tolist(),
        'scale': np.round(scale, 6).tolist(),
        'centroides': np.round(modelo.cluster_centers_[orden], 6).tolist(),
    }
    return np.array([remapeo[c] for c in etiquetas], dtype=int), geometria


def membresia(X_display, geometria):
    """Grupo de cada punto por centroide mas cercano; replica en numpy lo que hace el JS."""
    Z = ((aplicar_log(X_display, geometria['logs']) - np.array(geometria['offset']))
         / np.array(geometria['scale']))
    d = ((Z[:, None, :] - np.array(geometria['centroides'])[None, :, :]) ** 2).sum(axis=2)
    return d.argmin(axis=1)


# Extremos en unidades originales: el JS arma con esto la grilla del contorno. No dependen
# del espacio, porque las celdas son siempre las mismas; lo que cambia es la transformacion.
EXTENSION = [float(XY[:, 0].min()), float(XY[:, 0].max()),
             float(XY[:, 1].min()), float(XY[:, 1].max())]

GRUPOS_POR_ESPACIO, VANOS_POR_GRUPO, GEOMETRIAS = {}, {}, {}
for e, (log_x, log_y, prep) in enumerate(ESPACIOS):
    grupos, geometria = agrupar_celdas((log_x, log_y), prep)

    # El contorno se dibuja con la regla de centroide mas cercano, no con las etiquetas. Si
    # esa regla no reprodujera la particion de scikit-learn, la frontera mentiria sobre
    # donde termina cada grupo; se verifica celda por celda antes de embeberla.
    assert np.array_equal(membresia(XY, geometria), grupos),         f'la regla de centroide mas cercano no reproduce las etiquetas en el espacio {e}'

    GEOMETRIAS[str(e)] = geometria
    GRUPOS_POR_ESPACIO[str(e)] = grupos.tolist()

    # Vanos unicos de cada grupo: la union de los vanos de todas sus celdas. Un mismo vano
    # cuenta una sola vez aunque aparezca en varias ventanas del mismo grupo.
    asignado = CELDA_VANOS['celda'].map(_clave_celda)
    validas = asignado.notna()
    etiqueta_por_fila = pd.Series(grupos, index=range(len(CELDAS)))
    grupo_de_fila = asignado[validas].astype(int).map(etiqueta_por_fila)
    conteo = (CELDA_VANOS.loc[validas].assign(grupo=grupo_de_fila.values)
              .groupby('grupo')['FID_VANO'].nunique()
              .reindex(range(4)).fillna(0).astype(int).tolist())
    VANOS_POR_GRUPO[str(e)] = conteo

_g0 = np.array(GRUPOS_POR_ESPACIO[str(IDX_ESPACIO_DEFECTO)])
print(f'{len(CELDAS)} celdas con eventos | {len(ESPACIOS)} espacios agrupados')
print('espacio por defecto (lineal + minmax):')
for g, nombre in enumerate(NOMBRES_GRUPOS):
    print(f'  {nombre:<11s} celdas: {int((_g0 == g).sum()):>4d}   '
          f'vanos unicos: {VANOS_POR_GRUPO[str(IDX_ESPACIO_DEFECTO)][g]:>6,}   '
          f'mediana UITI: {np.median(XY[_g0 == g, 1]):>12,.1f}')

1738 celdas con eventos | 8 espacios agrupados
espacio por defecto (lineal + minmax):
  Bajo        celdas: 1455   vanos unicos: 23,377   mediana UITI:      2,777.1
  Medio       celdas:  236   vanos unicos: 12,671   mediana UITI:     27,643.3
  Medio-Alto  celdas:   17   vanos unicos:  1,992   mediana UITI:     65,084.4
  Alto        celdas:   30   vanos unicos:  1,912   mediana UITI:    310,626.6


In [5]:
import geopandas as gpd

# Misma geometria y mismo join que usa el mapa del reporte
# (`chec_local_interpreter.plotting.plot_circuit_map_folium`): las lineas de MVLINSEC.shp
# se cruzan por FID_VANO normalizado igual que `_norm_map_id`.
CLASES_MAPA = ['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto']
COLORES_MAPA = ['rgb(253,224,210)', 'rgb(252,146,114)', 'rgb(239,59,44)',
                'rgb(165,15,21)', 'rgb(103,0,13)']
COLOR_SIN_EVENTO = 'rgba(140,140,140,0.45)'


def _norm_id(serie):
    return (serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
            .replace({'': pd.NA, '<NA>': pd.NA, 'nan': pd.NA, 'None': pd.NA}))


_lineas = gpd.read_file(REPO_ROOT / 'data' / 'GEO' / 'MVLINSEC.shp')
if str(_lineas.crs) != 'EPSG:4326':
    _lineas = _lineas.to_crs('EPSG:4326')
_lineas['FID_VANO_GEO'] = _norm_id(_lineas['G3E_FID'])

_ev = pd.read_csv(REPO_ROOT / 'data' / 'Indicadores_vano_v3.csv',
                  usecols=['CIRCUITO', 'FID_VANO', 'UITI_VANO', 'FECHA'])
_ev['FID_VANO_NORM'] = _norm_id(_ev['FID_VANO'])
_ev['UITI_VANO'] = pd.to_numeric(_ev['UITI_VANO'], errors='coerce').fillna(0.0)
_ev['FECHA'] = pd.to_datetime(_ev['FECHA'], errors='coerce')

_con_geo = set(_lineas['FID_VANO_GEO'].dropna())
_utiles = _lineas[_lineas['FID_VANO_GEO'].isin(set(_ev['FID_VANO_NORM'].dropna()))]

# Geometria por circuito: cada vano es un segmento de dos puntos, asi que basta con sus
# extremos. Va una sola vez y no depende de la ventana; lo unico que cambia es el color.
GEO_POR_CIRCUITO = {}
for _circ, _grupo in _utiles.groupby(_utiles['CIRCUITO'].astype(str)):
    fids, lats, lons = [], [], []
    for _fid, _geom in zip(_grupo['FID_VANO_GEO'], _grupo.geometry):
        if _geom is None or _geom.is_empty:
            continue
        partes = [_geom] if _geom.geom_type == 'LineString' else list(getattr(_geom, 'geoms', []))
        for _parte in partes:
            xs, ys = _parte.xy
            fids.append(str(_fid))
            lats.append([round(v, 5) for v in ys])
            lons.append([round(v, 5) for v in xs])
    if fids:
        GEO_POR_CIRCUITO[_circ] = {'fids': fids, 'lat': lats, 'lon': lons,
                                   'centro': [round(float(np.mean([v for l in lats for v in l])), 5),
                                              round(float(np.mean([v for l in lons for v in l])), 5)]}

# UITI acumulado por vano y ventana, solo para vanos con geometria.
_ev_geo = _ev[_ev['FID_VANO_NORM'].isin(_con_geo)]
UITI_VENTANA_VANO = []
for _v in VENTANAS:
    _dentro = _ev_geo[(_ev_geo['FECHA'] >= _v['desde']) & (_ev_geo['FECHA'] < _v['hasta_excl'])]
    _suma = _dentro.groupby('FID_VANO_NORM')['UITI_VANO'].sum().round(3)
    UITI_VENTANA_VANO.append({str(k): float(x) for k, x in _suma.items()})

# Cortes de color FIJOS por circuito, calculados sobre todas sus ventanas a la vez. Si se
# recalcularan por ventana, mover el slider recolorearia el mapa aunque nada hubiera
# cambiado, y el movimiento aparente seria del normalizador, no del circuito.
UMBRALES_MAPA = {}
for _circ, _info in GEO_POR_CIRCUITO.items():
    _vals = [u[f] for f in set(_info['fids']) for u in UITI_VENTANA_VANO if f in u and u[f] > 0]
    if len(_vals) >= len(CLASES_MAPA):
        _q = np.quantile(_vals, np.linspace(0, 1, len(CLASES_MAPA) + 1)[1:-1])
        UMBRALES_MAPA[_circ] = [float(round(x, 4)) for x in _q]
    else:
        UMBRALES_MAPA[_circ] = [0.0] * (len(CLASES_MAPA) - 1)

print(f'{len(GEO_POR_CIRCUITO)} circuitos con geometria | '
      f'{sum(len(v["fids"]) for v in GEO_POR_CIRCUITO.values()):,} segmentos')
_tot_csv = _ev['FID_VANO_NORM'].nunique()
print(f'vanos del CSV con geometria: {len(set(_ev["FID_VANO_NORM"].dropna()) & _con_geo):,} '
      f'de {_tot_csv:,}')
print(f'celdas (vano, ventana) con UITI: {sum(len(u) for u in UITI_VENTANA_VANO):,}')

208 circuitos con geometria | 27,305 segmentos
vanos del CSV con geometria: 27,305 de 27,390
celdas (vano, ventana) con UITI: 110,759


In [6]:
# 18 trazas fijas: 1 contorno de membresia, 4 del mapa (una por grupo), 1 trayectoria,
# 2 del doble eje, 2 barras y 8 violines. El panel no crea ni destruye trazas, solo les
# reescribe los datos.
PERIODOS = [v['periodo'] for v in VENTANAS]
# Rotulos sin el ano: el rango completo se lee igual y ocupa la mitad, que es lo que evita
# que los ticks inclinados se metan en el titulo del panel de abajo. El periodo entero
# sigue disponible en el hover via customdata.
PERIODOS_CORTOS = [p.replace('2025-', '').replace('2026-', '') for p in PERIODOS]

fig = make_subplots(
    rows=7, cols=1,
    row_heights=[0.23, 0.14, 0.25, 0.09, 0.09, 0.10, 0.10], vertical_spacing=0.055,
    specs=[[{}], [{'secondary_y': True}], [{'type': 'map'}], [{}], [{}], [{}], [{}]],
    subplot_titles=('', 'Evolucion por ventana del circuito elegido',
                    'Mapa del circuito -- UITI acumulado en la ventana',
                    'Muestras por grupo', 'Vanos unicos por grupo',
                    'UITI acumulado por grupo', 'Numero de eventos por grupo'),
)

# Escala discreta de 4 escalones: cada banda del contorno toma el color de su grupo.
ESCALA_CONTORNO = []
for g, color in enumerate(COLORES_GRUPOS):
    ESCALA_CONTORNO.append([g / 4.0, color])
    ESCALA_CONTORNO.append([(g + 1) / 4.0, color])

fig.add_trace(go.Contour(                                     # traza 0: membresia de fondo
    z=[[0, 0], [0, 0]], x=[0, 1], y=[0, 1],
    colorscale=ESCALA_CONTORNO, zmin=-0.5, zmax=3.5, showscale=False,
    opacity=0.28, hoverinfo='skip', line=dict(width=1.2, color='rgba(120,20,20,0.6)'),
    contours=dict(start=-0.5, end=3.5, size=1, coloring='fill'),
    name='Membresia', showlegend=False,
), row=1, col=1)
for g in range(4):                                            # trazas 1-4: mapa por grupo
    fig.add_trace(go.Scattergl(
        x=[], y=[], mode='markers', name=NOMBRES_GRUPOS[g], legendgroup=NOMBRES_GRUPOS[g],
        marker=dict(size=5, color=COLORES_GRUPOS[g], opacity=OPACIDAD_NUBE),
        hovertext=[], hovertemplate='%{hovertext}<extra></extra>',
    ), row=1, col=1)
fig.add_trace(go.Scatter(                                     # traza 5: trayectoria elegida
    x=[], y=[], mode='lines+markers+text', name='Trayectoria', showlegend=False,
    line=dict(color='rgba(120,20,20,0.55)', width=1.5),
    marker=dict(size=14, color=[], line=dict(width=1.6, color='rgb(40,10,12)')),
    textposition='top center', textfont=dict(size=9, color='rgb(90,15,20)'),
    hovertext=[], hovertemplate='%{hovertext}<extra></extra>',
), row=1, col=1)

# Trazas 5-6: doble eje y. El UITI va contra el eje izquierdo y los eventos contra el
# derecho, porque viven en escalas muy distintas y compartir eje aplastaria uno de los dos.
fig.add_trace(go.Scatter(
    x=PERIODOS_CORTOS, y=[None] * len(VENTANAS), mode='lines+markers', name='UITI acumulado',
    line=dict(color='rgb(165,15,21)', width=2), marker=dict(size=7),
    customdata=PERIODOS,
    hovertemplate='%{customdata}<br>UITI acumulado: %{y:,.1f}<extra></extra>',
), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(
    x=PERIODOS_CORTOS, y=[None] * len(VENTANAS), mode='lines+markers', name='Numero de eventos',
    line=dict(color='rgb(251,106,74)', width=2, dash='dot'), marker=dict(size=7, symbol='square'),
    customdata=PERIODOS,
    hovertemplate='%{customdata}<br>Eventos: %{y:,}<extra></extra>',
), row=2, col=1, secondary_y=True)

fig.add_trace(go.Bar(                                         # traza 8: muestras por grupo
    x=NOMBRES_GRUPOS, y=[0] * 4, text=[0] * 4, textposition='outside',
    marker=dict(color=COLORES_GRUPOS, line=dict(width=0.5, color='rgba(60,10,10,0.6)')),
    showlegend=False, cliponaxis=False,
    hovertemplate='%{x}: %{y} muestras<extra></extra>',
), row=4, col=1)
fig.add_trace(go.Bar(                                         # traza 9: vanos unicos
    x=NOMBRES_GRUPOS, y=[0] * 4, text=[0] * 4, textposition='outside',
    marker=dict(color=COLORES_GRUPOS, line=dict(width=0.5, color='rgba(60,10,10,0.6)')),
    showlegend=False, cliponaxis=False,
    hovertemplate='%{x}: %{y} vanos unicos<extra></extra>',
), row=5, col=1)
for fila, etiqueta in [(6, 'UITI acumulado'), (7, 'Numero de eventos')]:  # trazas 10-17
    for g in range(4):
        fig.add_trace(go.Violin(
            x=[], y=[], name=NOMBRES_GRUPOS[g], legendgroup=NOMBRES_GRUPOS[g],
            showlegend=False, line=dict(color='rgba(90,15,20,0.85)', width=1),
            fillcolor=COLORES_GRUPOS[g], opacity=0.85,
            box_visible=True, meanline_visible=False, points=False, spanmode='hard',
            hovertemplate=f'%{{x}} -- {etiqueta}: %{{y:,.1f}}<extra></extra>',
        ), row=fila, col=1)

# Trazas 18-23 del mapa, agregadas AL FINAL a proposito: insertarlas antes correria los
# indices de todo lo anterior. Una traza por clase de color mas una gris para los vanos sin
# eventos en la ventana; Plotly no admite un color por segmento dentro de una misma traza.
for _c, (_clase, _color) in enumerate(zip(CLASES_MAPA, COLORES_MAPA)):
    fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=_clase, showlegend=False,
        line=dict(width=3, color=_color), hoverinfo='skip',
    ), row=3, col=1)
fig.add_trace(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Sin eventos', showlegend=False,
    line=dict(width=1.5, color=COLOR_SIN_EVENTO), hoverinfo='skip',
), row=3, col=1)

fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    title=dict(
        text='Trayectoria y agrupamiento de circuitos con ventana deslizante'
             '<br><sup>Cada punto es un par circuito x ventana; K-Means (k=4) sobre el espacio '
             'ajustado, grupos nombrados por la mediana del UITI</sup>',
        x=0.5, xanchor='center', yref='container', y=0.98, yanchor='top',
    ),
    legend=dict(title_text='', orientation='h', x=0.5, xanchor='center', y=1.015, yanchor='bottom'),
    margin=dict(t=140, r=90, b=60, l=90),
    height=2120, width=880, template='plotly_white', bargap=0.45, violingap=0.3,
)
fig.update_xaxes(title_text='Numero de eventos en la ventana', row=1, col=1)
fig.update_yaxes(title_text='UITI acumulado en la ventana', row=1, col=1)
fig.update_xaxes(tickangle=-30, tickfont=dict(size=9), row=2, col=1)
# Los rotulos del eje x van sin el ano para que entren, asi que el salto de ano deja de
# leerse. Una linea punteada entre la ultima ventana que arranca en un ano y la primera
# que arranca en el siguiente lo vuelve explicito. El eje es categorico: las posiciones
# son los indices, y la frontera entre la categoria k-1 y la k cae en k - 0.5.
# Se ancla a mano en vez de usar add_vline(row=, col=): con un subplot de tipo `map` en la
# misma figura, add_vline recorre todos los subplots y termina pasandole `xaxis` al
# Scattermap, que no tiene esa propiedad.
_ref_x_evo = 'x' + (fig.data[6].xaxis or 'x')[1:]
_ref_y_evo = 'y' + (fig.data[6].yaxis or 'y')[1:] + ' domain'
for _k in range(1, len(VENTANAS)):
    if VENTANAS[_k]['desde'].year != VENTANAS[_k - 1]['desde'].year:
        fig.add_shape(
            type='line', x0=_k - 0.5, x1=_k - 0.5, y0=0, y1=1,
            xref=_ref_x_evo, yref=_ref_y_evo,
            line=dict(dash='dot', width=1.5, color='rgba(60,60,60,0.75)'),
        )
        fig.add_annotation(
            # Adentro del panel, no arriba: en y=1 con anclaje inferior el rotulo se
            # montaba sobre el titulo del subplot y se leia "E2026ucion por ventana".
            x=_k - 0.5, y=0.97, xref=_ref_x_evo, yref=_ref_y_evo,
            text=str(VENTANAS[_k]['desde'].year), showarrow=False,
            xanchor='right', yanchor='top',
            font=dict(size=10, color='rgb(60,60,60)'),
        )
fig.update_yaxes(title_text='UITI acumulado', row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text='Eventos', row=2, col=1, secondary_y=True, showgrid=False)
fig.update_yaxes(title_text='Muestras', rangemode='tozero', row=4, col=1)
fig.update_yaxes(title_text='Vanos', rangemode='tozero', row=5, col=1)
fig.update_yaxes(title_text='UITI acumulado', row=6, col=1)
fig.update_yaxes(title_text='Eventos', row=7, col=1)
for _anotacion in fig.layout.annotations:
    _anotacion.font.size = 12

# Los ids de eje se leen de las trazas ya construidas en vez de hardcodearse: al agregar
# filas o un secondary_y la numeracion se corre, y un 'yaxis3' escrito a mano quedaria
# apuntando al panel equivocado sin que nada falle a la vista.
def _clave_eje(traza, cual):
    # OJO: `traza.y` son los DATOS; la referencia de eje vive en `traza.yaxis` ('y', 'y3'...).
    ref = getattr(traza, f'{cual}axis') or cual
    return f'{cual}axis' + ref[1:]


# Los indices de traza se declaran una sola vez y viajan al JS. Antes estaban escritos a
# mano en los dos lados, y al insertar el contorno adelante se habrian corrido todos sin
# que nada fallara a la vista: el panel escribiria en la traza equivocada.
IDX = {
    'contorno': 0,
    'mapa': [1, 2, 3, 4],
    'trayectoria': 5,
    'serieUiti': 6,
    'serieEventos': 7,
    'barrasMuestras': 8,
    'barrasVanos': 9,
    'violinUiti': [10, 11, 12, 13],
    'violinEventos': [14, 15, 16, 17],
    'mapaClases': [18, 19, 20, 21, 22],
    'mapaSinEventos': 23,
}
assert len(fig.data) == 24, len(fig.data)
assert all(fig.data[i].type == 'scattermap'
           for i in IDX['mapaClases'] + [IDX['mapaSinEventos']])
assert fig.data[IDX['contorno']].type == 'contour'
assert all(fig.data[i].type == 'scattergl' for i in IDX['mapa'])
assert all(fig.data[i].type == 'violin' for i in IDX['violinUiti'] + IDX['violinEventos'])
assert fig.data[IDX['barrasMuestras']].type == fig.data[IDX['barrasVanos']].type == 'bar'

EJES = {
    'mapaX': _clave_eje(fig.data[IDX['mapa'][0]], 'x'),
    'mapaY': _clave_eje(fig.data[IDX['mapa'][0]], 'y'),
    'barrasMuestras': _clave_eje(fig.data[IDX['barrasMuestras']], 'y'),
    'barrasVanos': _clave_eje(fig.data[IDX['barrasVanos']], 'y'),
    'violinUiti': _clave_eje(fig.data[IDX['violinUiti'][0]], 'y'),
    'violinEventos': _clave_eje(fig.data[IDX['violinEventos'][0]], 'y'),
}

# El print cierra la celda a proposito: si terminara en un update_*(), Jupyter mostraria
# la Figure devuelta y quedarian dos figuras, una de ellas sin panel de control.
print(f'{len(fig.data)} trazas: 1 contorno + 4 nube + 1 trayectoria + 2 doble eje '
      f'+ 2 barras + 8 violines + 6 mapa')
print('ejes resueltos:', EJES)

24 trazas: 1 contorno + 4 nube + 1 trayectoria + 2 doble eje + 2 barras + 8 violines + 6 mapa
ejes resueltos: {'mapaX': 'xaxis', 'mapaY': 'yaxis', 'barrasMuestras': 'yaxis4', 'barrasVanos': 'yaxis5', 'violinUiti': 'yaxis6', 'violinEventos': 'yaxis7'}


In [7]:
DIV_FIGURA = 'trayectorias-circuitos'

CONTEXTO = {
    'div': DIV_FIGURA,
    'circuitos': CIRCUITOS,
    'ventanas': [{'etiqueta': v['etiqueta'], 'periodo': v['periodo']} for v in VENTANAS],
    'porCircuito': POR_CIRCUITO,
    'espacios': [[bool(lx), bool(ly), prep] for lx, ly, prep in ESPACIOS],
    'grupos': NOMBRES_GRUPOS,
    'colores': COLORES_GRUPOS,
    'gruposPorEspacio': GRUPOS_POR_ESPACIO,
    'vanosPorGrupo': VANOS_POR_GRUPO,
    'sinSeleccion': SIN_SELECCION,
    'ejes': EJES,
    'idx': IDX,
    'geometrias': GEOMETRIAS,
    'extension': EXTENSION,
    'resolucion': 90,
    'opacidadNube': OPACIDAD_NUBE,
    'opacidadNubeAtenuada': OPACIDAD_NUBE_ATENUADA,
    'geo': GEO_POR_CIRCUITO,
    'uitiVentana': UITI_VENTANA_VANO,
    'umbrales': UMBRALES_MAPA,
    'clases': CLASES_MAPA,
}

_opciones = ''.join(f'<option value="{c}">{c}</option>' for c in CIRCUITOS)

PANEL_HTML = f'''
<style>
  .panel-tray {{
    font-family: system-ui, -apple-system, "Segoe UI", sans-serif; font-size: 13px;
    display: flex; flex-wrap: wrap; gap: 18px; align-items: flex-end;
    max-width: 860px; margin: 0 0 6px 0; padding: 12px 14px;
    border: 1px solid #e4c4c0; border-left: 4px solid rgb(203,24,29);
    border-radius: 6px; background: #fdf7f6; color: #2b2b2b;
  }}
  .panel-tray label {{ display: block; font-weight: 600; margin-bottom: 4px; }}
  .panel-tray select {{
    font: inherit; padding: 4px 6px; border: 1px solid #c9a9a5;
    border-radius: 4px; background: #fff; color: #2b2b2b; min-width: 140px;
  }}
  .panel-tray .chk {{ font-weight: 600; display: flex; align-items: center; gap: 6px; }}
  .panel-tray .chk input {{ margin: 0; }}
  .panel-tray .grupo-chk {{ display: flex; flex-direction: column; gap: 6px; }}
  .panel-tray button {{
    font: inherit; font-weight: 600; padding: 6px 12px; cursor: pointer;
    border: 1px solid rgb(203,24,29); border-radius: 4px;
    background: rgb(203,24,29); color: #fff;
  }}
  .panel-tray button:hover {{ background: rgb(165,15,21); }}
  .panel-aviso {{
    flex-basis: 100%; font-size: 12px; color: #7a5c58; margin: 0; font-weight: 400;
  }}
</style>
<div class="panel-tray">
  <div><label for="tr-circuito">Circuito</label>
       <select id="tr-circuito">
         <option value="{SIN_SELECCION}">{SIN_SELECCION}</option>{_opciones}
       </select></div>
  <div class="grupo-chk">
    <label class="chk"><input type="checkbox" id="tr-logx"> Log eje X (eventos)</label>
    <label class="chk"><input type="checkbox" id="tr-logy"> Log eje Y (UITI)</label>
  </div>
  <div><label for="tr-prep">Preproceso</label>
       <select id="tr-prep"><option value="minmax">minmax</option>
                            <option value="zscore">z-score</option></select></div>
  <div><button type="button" id="tr-csv">Descargar tabla (CSV)</button></div>
  <div style="flex-basis:100%; display:flex; align-items:center; gap:10px;">
    <label for="tr-ventana" style="margin:0; white-space:nowrap;">Ventana del mapa</label>
    <input type="range" id="tr-ventana" min="0" max="{len(VENTANAS) - 1}" value="0" step="1"
           style="flex:1; accent-color: rgb(203,24,29);">
    <span id="tr-ventana-txt" style="font-weight:600; white-space:nowrap; min-width:190px;"></span>
  </div>
  <p class="panel-aviso" id="tr-aviso"></p>
</div>
'''

PANEL_JS = '''
<script type="text/javascript">
(function () {
  var CTX = %s;
  var d = document;

  // Mismo recorrido canonico que CELDAS en Python: circuito y despues ventana, saltando
  // las celdas sin eventos. De aca sale el indice con que se leen las etiquetas de grupo.
  var CELDAS = [];
  CTX.circuitos.forEach(function (c, ci) {
    var reg = CTX.porCircuito[c];
    CTX.ventanas.forEach(function (v, vi) {
      if (reg.n[vi] > 0) { CELDAS.push([ci, vi]); }
    });
  });

  function etiquetaPunto(ci, vi, grupo) {
    var v = CTX.ventanas[vi], reg = CTX.porCircuito[CTX.circuitos[ci]];
    return '<b>' + CTX.circuitos[ci] + '</b> -- ' + v.etiqueta +
           '<br>' + v.periodo +
           '<br>Grupo: <b>' + CTX.grupos[grupo] + '</b>' +
           '<br>Eventos: ' + reg.n[vi] +
           '<br>UITI acumulado: ' + reg.u[vi];
  }

  function ejeGrilla(min, max, n, log) {
    // En logaritmica la grilla se reparte geometricamente, para que quede pareja en pantalla.
    var out = [], lo = log ? Math.log10(min) : min, hi = log ? Math.log10(max) : max;
    var paso = (hi - lo) / (n - 1);
    for (var i = 0; i < n; i++) {
      var v = lo + i * paso;
      out.push(log ? Math.pow(10, v) : v);
    }
    return out;
  }

  function contorno(geo) {
    // Membresia por centroide mas cercano en el espacio ajustado: la misma regla que
    // scikit-learn usa en predict(), verificada contra sus etiquetas del lado de Python.
    var ext = CTX.extension, n = CTX.resolucion;
    var lx = geo.logs[0], ly = geo.logs[1];
    var gx = ejeGrilla(ext[0], ext[1], n, lx);
    var gy = ejeGrilla(ext[2], ext[3], n, ly);
    var cen = geo.centroides, off = geo.offset, esc = geo.scale;
    var z = [];
    for (var j = 0; j < n; j++) {
      var ty = ((ly ? Math.log10(gy[j]) : gy[j]) - off[1]) / esc[1];
      var fila = [];
      for (var i = 0; i < n; i++) {
        var tx = ((lx ? Math.log10(gx[i]) : gx[i]) - off[0]) / esc[0];
        var mejor = 0, dmin = Infinity;
        for (var c = 0; c < cen.length; c++) {
          var a = tx - cen[c][0], b = ty - cen[c][1], dd = a * a + b * b;
          if (dd < dmin) { dmin = dd; mejor = c; }
        }
        fila.push(mejor);
      }
      z.push(fila);
    }
    return {z: z, x: gx, y: gy};
  }

  function csvTabla() {
    // Mismo esquema y mismo orden que tabla_ventanas() en Python: circuito y despues
    // ventana cronologica. La grilla va completa, con los ceros incluidos.
    var cab = ['circuito', 'ventana', 'desde', 'hasta', 'uiti_acumulado', 'num_eventos'];
    var out = [cab.join(',')];
    CTX.circuitos.forEach(function (c) {
      var reg = CTX.porCircuito[c];
      CTX.ventanas.forEach(function (v, k) {
        var partes = v.periodo.split(' a ');
        out.push(['"' + c + '"', v.etiqueta, partes[0], partes[1],
                  reg.u[k], reg.n[k]].join(','));
      });
    });
    return out.join('\\n') + '\\n';
  }

  function descargar() {
    var url = URL.createObjectURL(new Blob([csvTabla()],
                                           {type: 'text/csv;charset=utf-8;'}));
    var a = d.createElement('a');
    a.href = url; a.download = 'uiti_ventanas_deslizantes.csv';
    d.body.appendChild(a); a.click(); d.body.removeChild(a);
    setTimeout(function () { URL.revokeObjectURL(url); }, 0);
  }

  function claseDe(valor, umbrales) {
    if (!(valor > 0)) { return -1; }              // sin eventos en esta ventana
    for (var i = 0; i < umbrales.length; i++) {
      if (valor <= umbrales[i]) { return i; }
    }
    return umbrales.length;
  }

  function dibujarMapa(gd, circuito) {
    var w = parseInt(d.getElementById('tr-ventana').value, 10) || 0;
    var v = CTX.ventanas[w];
    d.getElementById('tr-ventana-txt').textContent = v.etiqueta + ': ' + v.periodo;

    var nClases = CTX.clases.length;
    var lat = [], lon = [], i;
    for (i = 0; i <= nClases; i++) { lat.push([]); lon.push([]); }   // el ultimo es "sin eventos"

    var info = (circuito !== CTX.sinSeleccion) ? CTX.geo[circuito] : null;
    if (info) {
      var uiti = CTX.uitiVentana[w], umbrales = CTX.umbrales[circuito] || [];
      for (i = 0; i < info.fids.length; i++) {
        var c = claseDe(uiti[info.fids[i]] || 0, umbrales);
        var destino = (c < 0) ? nClases : c;
        // Un null entre segmentos corta la linea: sin eso Plotly une el final de un vano
        // con el principio del siguiente y el mapa se llena de tramos que no existen.
        lat[destino] = lat[destino].concat(info.lat[i], [null]);
        lon[destino] = lon[destino].concat(info.lon[i], [null]);
      }
    }
    var indices = CTX.idx.mapaClases.concat([CTX.idx.mapaSinEventos]);
    Plotly.restyle(gd, {lat: lat, lon: lon}, indices);
    if (info) {
      Plotly.relayout(gd, {'map.center': {lat: info.centro[0], lon: info.centro[1]},
                           'map.zoom': 11});
    }
  }

  function aplicar() {
    var gd = d.getElementById(CTX.div);
    if (!gd || !gd._fullLayout) { return setTimeout(aplicar, 120); }

    var circuito = d.getElementById('tr-circuito').value;
    var logx = d.getElementById('tr-logx').checked;
    var logy = d.getElementById('tr-logy').checked;
    var prep = d.getElementById('tr-prep').value;
    var e = 0;
    for (var i = 0; i < CTX.espacios.length; i++) {
      var esp = CTX.espacios[i];
      if (esp[0] === logx && esp[1] === logy && esp[2] === prep) { e = i; break; }
    }
    var etiquetas = CTX.gruposPorEspacio[String(e)];

    // Mapa y violines: las celdas se reparten en los cuatro grupos del espacio elegido.
    var mx = [[], [], [], []], my = [[], [], [], []], mt = [[], [], [], []];
    var vg = [[], [], [], []], muestras = [0, 0, 0, 0];
    for (var k = 0; k < CELDAS.length; k++) {
      var ci = CELDAS[k][0], vi = CELDAS[k][1], g = etiquetas[k];
      var reg = CTX.porCircuito[CTX.circuitos[ci]];
      mx[g].push(reg.n[vi]); my[g].push(reg.u[vi]);
      mt[g].push(etiquetaPunto(ci, vi, g));
      vg[g].push(CTX.grupos[g]);
      muestras[g] += 1;
    }
    var ct = contorno(CTX.geometrias[String(e)]);
    Plotly.restyle(gd, {z: [ct.z], x: [ct.x], y: [ct.y]}, [CTX.idx.contorno]);
    Plotly.restyle(gd, {x: mx, y: my, hovertext: mt}, CTX.idx.mapa);
    Plotly.restyle(gd, {y: [muestras], text: [muestras]}, [CTX.idx.barrasMuestras]);
    Plotly.restyle(gd, {y: [CTX.vanosPorGrupo[String(e)]],
                        text: [CTX.vanosPorGrupo[String(e)]]}, [CTX.idx.barrasVanos]);
    Plotly.restyle(gd, {x: vg, y: my}, CTX.idx.violinUiti);
    Plotly.restyle(gd, {x: vg, y: mx}, CTX.idx.violinEventos);

    // Trayectoria del circuito elegido, con el color del grupo en que cayo cada ventana.
    var xs = [], ys = [], colores = [], textos = [], rotulos = [];
    // Series del doble eje: van las 11 ventanas, incluidas las de cero. Es una serie de
    // tiempo, y omitir los ceros haria parecer que esas ventanas no existieron.
    var serieU = [], serieN = [];
    if (circuito !== CTX.sinSeleccion) {
      var ci2 = CTX.circuitos.indexOf(circuito);
      var reg2 = CTX.porCircuito[circuito];
      serieU = reg2.u.slice(); serieN = reg2.n.slice();
      for (var k2 = 0; k2 < CELDAS.length; k2++) {
        if (CELDAS[k2][0] !== ci2) { continue; }
        var vi2 = CELDAS[k2][1], g2 = etiquetas[k2];
        xs.push(reg2.n[vi2]); ys.push(reg2.u[vi2]);
        colores.push(CTX.colores[g2]);
        textos.push(etiquetaPunto(ci2, vi2, g2));
        rotulos.push(CTX.ventanas[vi2].etiqueta);
      }
    } else {
      for (var k3 = 0; k3 < CTX.ventanas.length; k3++) { serieU.push(null); serieN.push(null); }
    }
    Plotly.restyle(gd, {y: [serieU]}, [CTX.idx.serieUiti]);
    Plotly.restyle(gd, {y: [serieN]}, [CTX.idx.serieEventos]);
    // Solo se rotula el arranque y el final: con 11 ventanas cercanas las etiquetas se
    // encimaban y se leian como una sola ("V1011"). El resto sale del hover.
    var visibles = rotulos.map(function (r, i) {
      return (i === 0 || i === rotulos.length - 1) ? r : '';
    });
    Plotly.restyle(gd, {x: [xs], y: [ys], 'marker.color': [colores],
                        text: [visibles], hovertext: [textos]}, [CTX.idx.trayectoria]);

    // Con un circuito elegido la nube de los otros 207 se atenua: su trayectoria son 11
    // puntos contra 1738, y a igual opacidad se pierde adentro. El contorno de membresia
    // se atenua junto con ella para que el fondo no gane peso al vaciarse la nube.
    var hay = circuito !== CTX.sinSeleccion;
    var opNube = hay ? CTX.opacidadNubeAtenuada : CTX.opacidadNube;
    Plotly.restyle(gd, {'marker.opacity': opNube}, CTX.idx.mapa);
    Plotly.restyle(gd, {opacity: hay ? 0.14 : 0.28}, [CTX.idx.contorno]);

    // Las flechas son anotaciones de layout, no una traza. En un eje logaritmico Plotly
    // espera las coordenadas YA en log10, asi que se convierten segun el tipo de cada eje.
    var refX = 'x' + CTX.ejes.mapaX.slice(5), refY = 'y' + CTX.ejes.mapaY.slice(5);
    var flechas = [];
    for (var i2 = 0; i2 + 1 < xs.length; i2++) {
      flechas.push({
        x: logx ? Math.log10(xs[i2 + 1]) : xs[i2 + 1],
        y: logy ? Math.log10(ys[i2 + 1]) : ys[i2 + 1],
        ax: logx ? Math.log10(xs[i2]) : xs[i2],
        ay: logy ? Math.log10(ys[i2]) : ys[i2],
        xref: refX, yref: refY, axref: refX, ayref: refY,
        showarrow: true, arrowhead: 3, arrowsize: 1.1, arrowwidth: 1.4,
        arrowcolor: 'rgba(120,20,20,0.75)', standoff: 10, startstandoff: 10, text: '',
      });
    }
    // Cada violin sigue el log de SU propia variable, no el del eje en que esta dibujado.
    // Las barras y el doble eje quedan siempre lineales: ahi hay ceros, que no existen en log.
    var tx = logx ? 'log' : 'linear', ty = logy ? 'log' : 'linear';
    var cambios = {annotations: fig_anotaciones.concat(flechas)};
    cambios[CTX.ejes.mapaX + '.type'] = tx;
    cambios[CTX.ejes.mapaY + '.type'] = ty;
    cambios[CTX.ejes.violinUiti + '.type'] = ty;
    cambios[CTX.ejes.violinEventos + '.type'] = tx;
    // Las barras rotulan por fuera: sin margen arriba, el numero de la barra mas alta se
    // mete en el titulo del panel. El margen sale del dato, no del autorango. Ojo: este
    // bloque se arma con formateo de cadena, asi que un simbolo de porcentaje suelto en un
    // comentario tira "not enough arguments for format string" al generar el cuaderno.
    var vanos = CTX.vanosPorGrupo[String(e)];
    cambios[CTX.ejes.barrasMuestras + '.range'] = [0, Math.max.apply(null, muestras) * 1.18];
    cambios[CTX.ejes.barrasVanos + '.range'] = [0, Math.max.apply(null, vanos) * 1.18];
    Plotly.relayout(gd, cambios);

    dibujarMapa(gd, circuito);

    var reparto = CTX.grupos.map(function (nombre, g) {
      return nombre + ' ' + CTX.vanosPorGrupo[String(e)][g];
    }).join(' / ');
    d.getElementById('tr-aviso').textContent = (circuito === CTX.sinSeleccion
      ? 'Elegi un circuito para ver su trayectoria. '
      : circuito + ': ' + xs.length + ' de ' + CTX.ventanas.length +
        ' ventanas con eventos, ' + flechas.length + ' tramos. ') +
      'Vanos unicos por grupo -- ' + reparto + '.';
  }

  // Los titulos de los subplots tambien son anotaciones: hay que conservarlos al
  // reescribir `annotations` con las flechas, o desaparecen en el primer cambio.
  var fig_anotaciones = (function () {
    var gd = d.getElementById(CTX.div);
    return (gd && gd.layout && gd.layout.annotations) ? gd.layout.annotations.slice() : [];
  })();

  ['tr-circuito', 'tr-logx', 'tr-logy', 'tr-prep'].forEach(function (id) {
    var el = d.getElementById(id);
    if (el) { el.addEventListener('change', aplicar); }
  });
  var boton = d.getElementById('tr-csv');
  if (boton) { boton.addEventListener('click', descargar); }
  // El slider solo repinta el mapa: no cambia la particion ni ninguna otra figura.
  var slider = d.getElementById('tr-ventana');
  if (slider) {
    slider.addEventListener('input', function () {
      var gd = d.getElementById(CTX.div);
      if (gd && gd._fullLayout) {
        dibujarMapa(gd, d.getElementById('tr-circuito').value);
      }
    });
  }
  aplicar();
  // MapLibre inicializa de forma asincrona: un restyle/relayout disparado antes de que el
  // subplot de mapa este listo se pierde en silencio -- el fondo carga pero las lineas del
  // circuito no aparecen. Se repite el dibujado un par de veces despues del arranque; es
  // idempotente, asi que repetirlo no tiene costo mas alla de esas dos pasadas.
  [700, 2000].forEach(function (ms) {
    setTimeout(function () {
      var gd = d.getElementById(CTX.div);
      var sel = d.getElementById('tr-circuito');
      if (gd && gd._fullLayout && sel) { dibujarMapa(gd, sel.value); }
    }, ms);
  });
})();
</script>
''' % json.dumps(CONTEXTO, separators=(',', ':'))

# include_plotlyjs=True embebe plotly.js en esta misma salida: el panel, la figura y su
# libreria viajan juntos, asi el cuaderno se ve igual exportado a HTML o en nbviewer.
FIGURA_HTML = pio.to_html(fig, include_plotlyjs=True, full_html=False, div_id=DIV_FIGURA)

display(HTML(PANEL_HTML + FIGURA_HTML + PANEL_JS))

In [8]:
def guardar_tabla(destino=None):
    """Escribe TABLA en reports/interpretability/artifacts/.

    Es el camino reproducible del boton "Descargar tabla (CSV)": mismo esquema y mismo
    orden, para que el archivo que baja el navegador y el que escribe el kernel coincidan.
    """
    if destino is None:
        destino = (REPO_ROOT / 'reports' / 'interpretability' / 'artifacts' /
                   'uiti_ventanas_deslizantes.csv')
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    TABLA.to_csv(destino, index=False)
    return destino


ruta_csv = guardar_tabla()
print(f'{len(TABLA)} filas -> {ruta_csv.relative_to(REPO_ROOT)}')

# Circuitos con mas movimiento entre ventanas: los que mas se van a notar en el mapa.
resumen = (TABLA.groupby('circuito')
           .agg(ventanas_con_eventos=('num_eventos', lambda s: int((s > 0).sum())),
                uiti_total=('uiti_acumulado', 'sum'),
                uiti_max_ventana=('uiti_acumulado', 'max'))
           .sort_values('uiti_total', ascending=False))
resumen.head(10)

2288 filas -> reports/interpretability/artifacts/uiti_ventanas_deslizantes.csv


,ventanas_con_eventos,uiti_total,uiti_max_ventana
circuito,,,
AZA23L17,8,1309267.20,622369.05
DON23L12,10,1271141.47,630011.11
BQE23L12,11,1178697.99,442279.93
CHA23L14,11,1035702.45,344202.08
PSO23L12,11,848728.09,318599.43
SCH23L12,10,825955.58,325078.27
AZA23L19,6,806569.10,396113.55
NSA23L14,11,782807.47,219700.93
HER23L16,11,776677.84,208852.53


## Como leerlo

- Un punto es un par **circuito x ventana**, no un circuito: el mismo circuito aparece hasta 11
  veces, una por ventana, y **puede caer en grupos distintos** segun la ventana. Eso es lo que
  hace legible la trayectoria: se ve a un circuito pasar de `Bajo` a `Alto` y volver.
- Las ventanas **se solapan de a 14 a 17 dias** a proposito. Dos puntos consecutivos comparten
  cerca de la mitad de sus datos, asi que la trayectoria se mueve suave por construccion: un salto
  grande entre ventanas vecinas senala un evento fuerte concentrado en los dias que no comparten.
  La contracara es que los puntos consecutivos **no son independientes**.
- La secuencia alterna **mes calendario completo** y **cruzada** (del 15 al 15). Las de mes
  completo son las comparables entre si mes a mes; las cruzadas revelan si un pico quedo partido
  entre dos meses.
- Las **flechas** apuntan siempre de la ventana mas vieja a la mas nueva, y el color de cada
  marcador es el del grupo en que cayo esa ventana.
- El **conteo de vanos unicos** no es proporcional al de celdas: un grupo con pocas celdas puede
  contener circuitos grandes y sumar mas vanos que otro con muchas celdas chicas. Un vano se cuenta
  una sola vez por grupo aunque aparezca en varias de sus ventanas.
- Los **violines** muestran la distribucion completa de cada variable dentro de cada grupo, con su
  caja y su mediana -- el estadistico con el que se ordenan los nombres.

**Limitacion.** El corte en el dia 15, el ancho de un mes y `k=4` son decisiones de agregacion, no
algo que impongan los datos. El solape hace que los puntos consecutivos no sean independientes, asi
que esta figura sirve para leer el recorrido de un circuito, no para estimar tendencias formales.